# 1. Batch Correction of RNA Embeddings

Apply batch correction methods (Harmony + DANN) to Foundation Model RNA embeddings.

**Pipeline:**
1. Load & concatenate cohorts
2. Baseline UMAP (before correction)
3. Harmony stage-1 + BackTrack stage-2
4. DANN adversarial correction
5. Comparison & save corrected AnnData

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt

from batchcor_rna_emb.logging_config import set_logger
from batchcor_rna_emb.config import SEED, DATA_DIR, DATA_INTERIM_DIR, FIGURES_DIR, BATCH_COL, DIAGNOSIS_COL
from batchcor_rna_emb.data_io import load_all_cohorts, save_adata_zarr

# Loguru setup — call once before any other imports that log
set_logger(level="INFO")

# Reproducibility
np.random.seed(SEED)

In [ ]:
# Scanpy / UMAP figure params
sc.settings.figdir = str(FIGURES_DIR)
sc.set_figure_params(dpi=150, dpi_save=300, figsize=(5, 5))

UMAP_DIST = 0.4
UMAP_SPREAD = 1.0

# Ensure output directories exist
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DATA_INTERIM_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Cohort Data

In [ ]:
# Load all cohorts from data/ directory
cohorts = load_all_cohorts(DATA_DIR)

from loguru import logger
for c in cohorts:
    logger.info("Cohort: {} | batch values: {}", c.shape, c.obs[BATCH_COL].value_counts().to_dict() if BATCH_COL in c.obs.columns else "N/A")

In [ ]:
# Concatenate cohorts
adata = ad.concat(cohorts, join="outer")
logger.info("Combined AnnData: {} samples x {} genes", adata.n_obs, adata.n_vars)
logger.info("Batch distribution:\n{}", adata.obs[BATCH_COL].value_counts())
logger.info("Diagnosis distribution:\n{}", adata.obs[DIAGNOSIS_COL].value_counts())

# Identify embedding keys in .obsm
emb_keys = [k for k in adata.obsm.keys() if k.startswith("FM_")]
logger.info("Available embedding keys: {}", emb_keys)

# Select embedding to correct (use first FM embedding found)
EMB_KEY = emb_keys[0] if emb_keys else None
logger.info("Using embedding key: '{}'", EMB_KEY)

In [ ]:
# Split into train/test using is_train or Split_ column
# Adapt to your data: check which split column exists
split_col_candidates = [c for c in adata.obs.columns if c.startswith("Split_") or c == "is_train"]
logger.info("Available split columns: {}", split_col_candidates)

# Create split column if needed
if "split" not in adata.obs.columns:
    if "is_train" in adata.obs.columns:
        adata.obs["split"] = np.where(adata.obs["is_train"], "train", "test")
    elif split_col_candidates:
        first_split = split_col_candidates[0]
        adata.obs["split"] = adata.obs[first_split].astype(str)
    else:
        logger.warning("No split column found — using random 70/30 split")
        rng = np.random.RandomState(SEED)
        adata.obs["split"] = np.where(rng.rand(adata.n_obs) < 0.7, "train", "test")

adata_train = adata[adata.obs["split"] == "train"].copy()
adata_test = adata[adata.obs["split"] == "test"].copy()
logger.info("Train: {}, Test: {}", adata_train.n_obs, adata_test.n_obs)

## 2. Baseline Visualization (Before Correction)

In [ ]:
# Compute UMAP on raw (uncorrected) embeddings — train only
sc.pp.neighbors(adata_train, use_rep=EMB_KEY, n_neighbors=30, metric="cosine")
sc.tl.umap(adata_train, min_dist=UMAP_DIST, spread=UMAP_SPREAD)

sc.pl.umap(adata_train, color=BATCH_COL, s=5, title="Raw Embeddings — Batch", show=True)
sc.pl.umap(adata_train, color=DIAGNOSIS_COL, s=5, title="Raw Embeddings — Diagnosis", show=True)

## 3. Harmony Batch Correction

**Stage-1:** Run Harmony on train embeddings to correct for batch effects.
**Stage-2 (BackTrack):** Project test onto frozen train UMAP via barycentric coordinates.

In [ ]:
from batchcor_rna_emb.batch_correction.harmony import run_harmony_stage1

# Stage-1: correct train embeddings
HARMONY_KEY = f"{EMB_KEY}_Harmony"
corrected_train = run_harmony_stage1(adata_train, embedding_key=EMB_KEY, batch_col=BATCH_COL)
adata_train.obsm[HARMONY_KEY] = corrected_train
logger.info("Harmony stage-1 output stored in obsm['{}']", HARMONY_KEY)

In [ ]:
# Elbow plot: determine n_pcs for Harmony-corrected embeddings
harmony_pcs = adata_train.obsm[HARMONY_KEY]
variance = np.var(harmony_pcs, axis=0)
variance_sorted = np.sort(variance)[::-1][:80]

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(variance_sorted) + 1), variance_sorted, "o-", color="b", markersize=3)
plt.title("Elbow Plot: Harmony-corrected Embeddings")
plt.xlabel("Component")
plt.ylabel("Variance")
plt.grid(True)
plt.tight_layout()
plt.show()

# Determine n_pcs visually — adjust as needed
N_PCS_HARMONY = 42
logger.info("Using n_pcs={} for Harmony UMAP", N_PCS_HARMONY)

In [ ]:
# Sort Harmony embeddings by variance (descending) before UMAP
emb_matrix = adata_train.obsm[HARMONY_KEY]
variances = np.var(emb_matrix, axis=0)
desc_idx = np.argsort(variances)[::-1]
adata_train.obsm[HARMONY_KEY] = emb_matrix[:, desc_idx]

# Compute train UMAP on Harmony-corrected embeddings
sc.pp.neighbors(adata_train, use_rep=HARMONY_KEY, n_pcs=N_PCS_HARMONY, n_neighbors=30, metric="cosine")
sc.tl.umap(adata_train, min_dist=UMAP_DIST, spread=UMAP_SPREAD, key_added="UMAP_Harmony_")

sc.pl.embedding(adata_train, basis="UMAP_Harmony_", color=BATCH_COL, s=5, title="Harmony Stage-1 — Batch")
sc.pl.embedding(adata_train, basis="UMAP_Harmony_", color=DIAGNOSIS_COL, s=5, title="Harmony Stage-1 — Diagnosis")

In [ ]:
from batchcor_rna_emb.batch_correction.harmony import backtrack_harmony_integration, robustness_index

# BackTrack Harmony Integration: stage-2 + barycentric projection
combined, diag = backtrack_harmony_integration(
    adata_train, adata_test,
    embedding_key=EMB_KEY,
    batch_col=BATCH_COL,
    split_col="split",
    umap_train_key="UMAP_Harmony_",
    qc_neighbors=30,
    qc_metric="cosine",
    ood_q=0.995,
    ood_factor=1.0,
    ood_mode="stage2_knn",
    k_fallback=30,
)

logger.info("BackTrack diagnostics: {}", diag)

In [ ]:
# Post-Harmony UMAP: combined train+test
sc.pl.embedding(combined, basis="X_umap", color="split", s=5, title="BackTrack Harmony — Split")
sc.pl.embedding(combined, basis="X_umap", color=BATCH_COL, s=5, title="BackTrack Harmony — Batch")
sc.pl.embedding(combined, basis="X_umap", color=DIAGNOSIS_COL, s=5, title="BackTrack Harmony — Diagnosis")

In [ ]:
# QC: Robustness Index (RI = SO / (SO + OS))
ri_all, ri_details = robustness_index(
    combined,
    emb_key="X_umap",
    bio_key=DIAGNOSIS_COL,
    conf_key="combined_batch",
    k=50,
)
logger.info("Robustness Index: {:.3f}, details: {}", ri_all, ri_details)

## 4. DANN Batch Correction

**Domain Adversarial Neural Network (DANN)**:
- Encoder maps embeddings to batch-invariant latent space
- Gradient Reversal Layer (GRL) makes batch discriminator adversarial
- Optional bio classifier preserves diagnosis signal
- Progressive λ warmup over first 30% of epochs

Architecture: `Input → [512→BN→ReLU→Drop] → [256→BN→ReLU] → latent_dim`
Loss: `L_recon + λ_adv * L_adv - λ_bio * L_bio`

In [ ]:
from batchcor_rna_emb.batch_correction.dann import DANNCorrector, DANNConfig

dann_config = DANNConfig(
    latent_dim=256,
    n_epochs=100,
    batch_size=256,
    lr=1e-3,
    lambda_adv=1.0,
    lambda_bio=0.5,
    warmup_fraction=0.3,
    dropout=0.3,
    seed=SEED,
)

dann = DANNCorrector(config=dann_config)

# Fit on train embeddings
X_train_emb = np.asarray(adata_train.obsm[EMB_KEY], dtype=np.float32)
batch_labels = adata_train.obs[BATCH_COL].to_numpy()
bio_labels = adata_train.obs[DIAGNOSIS_COL].to_numpy() if DIAGNOSIS_COL in adata_train.obs.columns else None

dann.fit(X_train_emb, batch_labels, bio_labels)

In [ ]:
# Transform train and test embeddings
DANN_KEY = f"{EMB_KEY}_DANN"

adata_train.obsm[DANN_KEY] = dann.transform(X_train_emb)
X_test_emb = np.asarray(adata_test.obsm[EMB_KEY], dtype=np.float32)
adata_test.obsm[DANN_KEY] = dann.transform(X_test_emb)

logger.info("DANN latent stored in obsm['{}']: train={}, test={}",
            DANN_KEY, adata_train.obsm[DANN_KEY].shape, adata_test.obsm[DANN_KEY].shape)

In [ ]:
# Post-DANN UMAP
sc.pp.neighbors(adata_train, use_rep=DANN_KEY, n_neighbors=30, metric="cosine")
sc.tl.umap(adata_train, min_dist=UMAP_DIST, spread=UMAP_SPREAD, key_added="UMAP_DANN_")

sc.pl.embedding(adata_train, basis="UMAP_DANN_", color=BATCH_COL, s=5, title="DANN — Batch")
sc.pl.embedding(adata_train, basis="UMAP_DANN_", color=DIAGNOSIS_COL, s=5, title="DANN — Diagnosis")

In [ ]:
# DANN loss curves
from batchcor_rna_emb.visualization.plots import plot_loss_curves

history_dict = {
    "total": dann.history_.total,
    "recon": dann.history_.recon,
    "adversarial": dann.history_.adv,
    "bio": dann.history_.bio,
    "lambda": dann.history_.lambda_schedule,
}
plot_loss_curves(history_dict, title="DANN Training Loss", save_path=str(FIGURES_DIR / "dann_loss_curves.png"))

## 5. Comparison & Save

In [ ]:
# 2x3 UMAP comparison grid: Raw / Harmony / DANN × batch / diagnosis
from batchcor_rna_emb.visualization.plots import plot_umap_grid

# Build combined AnnData for each method (train only for fair comparison)
adata_raw = adata_train.copy()
adata_harmony = adata_train.copy()
adata_dann = adata_train.copy()

# Compute UMAP for raw (already done, in X_umap)
# Harmony UMAP already in UMAP_Harmony_
# DANN UMAP already in UMAP_DANN_

# Standardize to X_umap for the grid
adata_raw.obsm["X_umap"] = adata_raw.obsm["X_umap"]  # already set
adata_harmony.obsm["X_umap"] = adata_harmony.obsm["UMAP_Harmony_"]
adata_dann.obsm["X_umap"] = adata_dann.obsm["UMAP_DANN_"]

grid_adatas = {"Raw": adata_raw, "Harmony": adata_harmony, "DANN": adata_dann}
fig = plot_umap_grid(
    grid_adatas,
    color_cols=[BATCH_COL, DIAGNOSIS_COL],
    save_path=str(FIGURES_DIR / "umap_comparison_grid.png"),
)
plt.show()

In [ ]:
# Save corrected AnnData: recombine train + test with all obsm keys
adata_corrected = ad.concat([adata_train, adata_test], join="outer")
adata_corrected.obs["split"] = pd.Categorical(
    list(adata_train.obs["split"]) + list(adata_test.obs["split"])
)

output_path = DATA_INTERIM_DIR / "corrected.adata.zarr"
save_adata_zarr(adata_corrected, output_path)

# Summary of .obsm keys
logger.info("Saved corrected AnnData .obsm keys: {}", list(adata_corrected.obsm.keys()))